In [1]:
import pandas as pd
import numpy as np
import warnings
from micom.workflows import load_results, GrowthResults
from micom.stats import compare_groups
import sys
import math

sys.path.append('..')
from utils import analysis

In [2]:
res = load_results('../goll_et.al_2020_IBS_FMT/models/res_western.zip')
ex_t0 = res.exchanges[(res.exchanges.sample_id.str.startswith('D') | res.exchanges.sample_id.str.endswith('-0'))].copy()
growth_t0 = res.growth_rates[(res.growth_rates.sample_id.str.startswith('D') | res.growth_rates.sample_id.str.endswith('-0'))].copy()
res_t0 = GrowthResults(growth_rates=growth_t0, exchanges=ex_t0, annotations=res.annotations.copy())
res_t0.exchanges['sample'] = res_t0.exchanges['sample_id'].apply(lambda x: 'donor' if x.startswith('D') else 'recipient')
res_t0.growth_rates['sample'] = res_t0.growth_rates['sample_id'].apply(lambda x: 'donor' if x.startswith('D') else 'recipient')

In [3]:
t0_production_rates = analysis.production_rates_with_imputation(res_t0)
t0_production_rates['sample'] = t0_production_rates['sample_id'].apply(lambda x: 'donor' if x.startswith('D') else 'recipient')
total_t0_production = compare_groups(t0_production_rates, metadata_column='sample', groups=['donor', 'recipient'], threads=12, progress=True)

total_t0_production = (
    total_t0_production.merge(
        res_t0.annotations[['metabolite', 'name']],
        on='metabolite',
        how='left'
    )
    .assign(
        **{'-log10q_val': lambda x: -np.log10(x['q'])}
    )
)

total_t0_production['enrichment'] = total_t0_production.apply(lambda x: 'enriched' if (x['log_fold_change'] > 0 and x['q'] < 0.05) else ('depleted' if (x['log_fold_change'] < 0 and x['q'] < 0.05) else 'n.s.'), axis=1)
total_t0_production.to_csv('./data/2A.csv')

Output()

In [4]:
metabolites_to_plot = {
    'Formate': 'for[e]', 
    'acetate': 'ac[e]', 
    'propionate': 'ppa[e]', 
    'butyrate': 'but[e]', 
    'Valeric Acid': 'M03134[e]', 
    'Sulfite': 'so3[e]', 
    'thiosulfate(2-)': 'tsul[e]', 
    'Methanethiol': 'ch4s[e]',
    'Hydrogen': 'h2[e]', 
}

ex_metabolites = res_t0.exchanges[res_t0.exchanges.metabolite.isin(metabolites_to_plot.values())].copy()
ex_metabolites = ex_metabolites.query('taxon != "medium" and direction == "export"')
ex_metabolites['flux'] = ex_metabolites['flux'] * ex_metabolites['abundance']
ex_metabolites = (
    ex_metabolites
    .groupby(['sample_id', 'metabolite'], as_index=False)
    .agg({'flux': 'sum'})
)

unique_samples = ex_metabolites['sample_id'].unique()
unique_metabolites = list(metabolites_to_plot.values())

template = pd.MultiIndex.from_product(
    [unique_samples, unique_metabolites],
    names=['sample_id', 'metabolite']
).to_frame(index=False)

template['sample'] = template['sample_id'].apply(lambda x: 'donor' if x.startswith('D') else 'recipient')

ex_metabolites_complete = template.merge(
    ex_metabolites[['sample_id', 'metabolite', 'flux']], 
    on=['sample_id', 'metabolite'],
    how='left'  
).fillna(0)

id2metabolites = res_t0.annotations.set_index('metabolite')['name']
ex_metabolites_complete['name'] = ex_metabolites_complete['metabolite'].map(id2metabolites)

ex_metabolites_complete.to_csv('./data/2B.csv')

In [5]:
abundance_raw = pd.read_csv('../goll_et.al_2020_IBS_FMT/raw/goll_agora_map.counts.csv').set_index('taxon_id')
abundance_normalized_by_genome_length = pd.read_csv('../goll_et.al_2020_IBS_FMT/processed/abundance_normalized_table.csv').set_index('taxon_id')
DonorRecipientMapping = pd.read_csv('../goll_et.al_2020_IBS_FMT/raw/DonorRecipientMapping.csv')

donors = DonorRecipientMapping['donor'].str.split('_').str[0].unique().tolist()
recipients_T0 = DonorRecipientMapping['recipient'].str.split('_').str[0].unique().tolist()
recipients_T6 = [x.replace('-0', '-6') for x in recipients_T0]

recipients_T6 = set(recipients_T6).intersection(set(abundance_normalized_by_genome_length.columns))
recipients_T0 = [x.replace('-6', '-0') for x in recipients_T6]

all_samples = donors + recipients_T0 + list(recipients_T6)
final_samples = set(all_samples).intersection(set(abundance_normalized_by_genome_length.columns))

abundance_raw = abundance_raw.loc[:, abundance_raw.columns.isin(final_samples)]
abundance_normalized_by_genome_length = abundance_normalized_by_genome_length.loc[:, abundance_normalized_by_genome_length.columns.isin(final_samples)]

abundance_raw.to_csv('./data/abundance_raw_subset_unfiltered.csv')
abundance_normalized_by_genome_length.to_csv('./data/abundance_normalized_subset_unfiltered.csv')

abundance_normalized_by_genome_length_filtered = abundance_normalized_by_genome_length[(((abundance_normalized_by_genome_length / abundance_normalized_by_genome_length.sum(axis=0)) >= 0.0001).sum(axis=1) >= math.ceil((0.05 * abundance_normalized_by_genome_length.shape[1])))]
abundance_normalized_by_genome_length_filtered.to_csv('./data/abundance_normalized_subset_filtered.csv')

abundance_raw_filtered = abundance_raw.loc[abundance_raw.index.isin(abundance_normalized_by_genome_length_filtered.index), :]
abundance_raw_filtered.to_csv('./data/abundance_raw_subset_filtered.csv')

In [6]:
recipients = list(recipients_T6) + recipients_T0 + donors

growth_t6 = res.growth_rates[res.growth_rates.sample_id.isin(recipients)]
ex_t6 = res.exchanges[res.exchanges.sample_id.isin(recipients)]
res_t6 = GrowthResults(growth_rates=growth_t6, exchanges=ex_t6, annotations=res.annotations)

SCFA_to_plot = {
    'Formate': 'for[e]', 
    'acetate': 'ac[e]', 
    'propionate': 'ppa[e]', 
    'butyrate': 'but[e]', 
    'Valeric Acid': 'M03134[e]', 
}

DonorRecipientMapping = pd.read_csv('../goll_et.al_2020_IBS_FMT/raw/DonorRecipientMapping.csv')
DonorRecipientMapping['donor'] = DonorRecipientMapping['donor'].str.split('_').str[0]
DonorRecipientMapping['recipient t0'] = DonorRecipientMapping['recipient'].str.split('_').str[0]
DonorRecipientMapping['recipient t6'] = DonorRecipientMapping['recipient t0'].str.replace('-0', '-6')
DonorRecipientMapping = DonorRecipientMapping.loc[
    DonorRecipientMapping['recipient t6'].isin(list(recipients_T6)), :
]

mapping = (
    DonorRecipientMapping[['clinical_response', 'recipient t0', 'recipient t6']]
    .melt(
        id_vars='clinical_response',
        value_vars=['recipient t0', 'recipient t6'],
        value_name='recipients'
    )
    .drop(columns=['variable'])
)

donor = (
    pd.DataFrame(
        'donor', 
        index=donors, 
        columns=['clinical_response']
    )
    .reset_index()
    .rename(columns={'index': 'recipients'})
)

mapping = pd.concat([mapping, donor], axis=0, ignore_index=True)
t6_production_rates = (
    analysis.production_rates_with_imputation(res_t6)
    .merge(
        mapping,
        left_on='sample_id',
        right_on='recipients',
        how='left'
    )
    .query('metabolite.isin(@SCFA_to_plot.values())')
    .assign(
        time=lambda x: np.where(
            x['sample_id'].str.endswith('-6'),
            'T6',
            'T0'
        )
    )
    .groupby(
        ['sample_id', 'time', 'clinical_response'],
        as_index=False
    )['flux'].sum()
)
t6_production_rates['base_sample_id'] = t6_production_rates['sample_id'].apply(lambda x: x if x.startswith('D') else x.split('-')[0])
t6_production_rates.to_csv('./data/2E.csv')